In [ ]:
import os
import logging
import subprocess

from tqdm import tqdm
from pathlib import Path
from datasets import load_dataset

from concurrent.futures import ThreadPoolExecutor

In [8]:
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [9]:
AUDIO_DIR = Path("/content/drive/MyDrive/KPI/audio/Lab-3")

N_SAMPLES = 10000

In [4]:
dataset = load_dataset(
    "nvidia/Granary",
    "uk",
    split="ast"
)

README.md: 0.00B [00:00, ?B/s]

uk/ytc/uk_asr.jsonl:   0%|          | 0.00/2.82M [00:00<?, ?B/s]

uk/ytc/uk_ast-en.jsonl:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

Generating asr split: 0 examples [00:00, ? examples/s]

Generating ast split: 0 examples [00:00, ? examples/s]

In [5]:
print(dataset[0])

{'audio_filepath': 'ytc/uk/jC2Q_T89YXI-268552-240.wav', 'text': 'Комічне.', 'duration': 2.4, 'source_lang': 'uk', 'target_lang': 'en', 'taskname': 'ast', 'utt_id': 'jC2Q_T89YXI-268552-240', 'original_source_id': 'jC2Q_T89YXI-268552-240', 'dataset_source': 'ytc', 'offset': 2685.52, 'answer': 'Comical.'}


In [6]:
video_ids = list({item["utt_id"].split("-")[0] for item in dataset})
print("Unique videos:", len(video_ids))

Unique videos: 44


In [12]:
video_ids[:5]

['vobL48IvGNM', 'MrRj8pPBk8Q', 'iTInWOM55EI', 'Pv6d4jEdtg8', 'jdyymgviChA']

In [ ]:
video_ids[22] = "ChjeZ4a7-VU"
video_ids[43] = "ot1w-MD4qFs"

In [50]:
video_ids

['vobL48IvGNM',
 'MrRj8pPBk8Q',
 'iTInWOM55EI',
 'Pv6d4jEdtg8',
 'jdyymgviChA',
 '9h4gVMxlEuU',
 'WJ3Lswb5OXA',
 'Sdu3Br1astQ',
 '0tPw2WObLwg',
 'cO6iTTb6C0Y',
 'HhwIpERWpgQ',
 'eIX04amSZ_M',
 'tq3MUTO7B2M',
 'TXpT9AiBJyE',
 '9eXtf1lN_G8',
 'G0yfpbiXgpM',
 'LdiWukY3Wm0',
 'YtfG95TcUcM',
 'cET3iqno4LE',
 '3z8lMpOx5qE',
 '05ctuwHm60U',
 'Nyirzhhg880',
 'ChjeZ4a7-VU',
 '7jVEKYMcy6o',
 'Vv1BMT5atro',
 'jC2Q_T89YXI',
 'fUKFIBSGKVs',
 'D_ZMV0hld1w',
 '_GMVKJ3Jf44',
 'Y___XPvO7e8',
 '56aNsdB3Bvs',
 'PTAuIyHhCEk',
 'iXSQ8zpInko',
 'UWcjBajvlDQ',
 'fWaomxp7tXU',
 'JOxRHUfKSWc',
 'Q8jhvJaodNU',
 '2nhhRpDKXo8',
 'YlsgLP3G7Aw',
 'Cz_3tsRtJBU',
 '6pXDgw3dTcA',
 'vxHKM08mcWo',
 'VGSeCbxNuoA',
 'ot1w-MD4qFs']

In [40]:
os.makedirs("videos", exist_ok=True)

def download_video(video_id):
    out = f"videos/{video_id}.wav"
    
    if any(fname.startswith(video_id) for fname in os.listdir("videos")):
        return True

    url = f"https://www.youtube.com/watch?v={video_id}"

    try:
        cmd = [
            "yt-dlp",
            url,
            "--extract-audio",
            "--audio-format", "wav",
            "--output", out,
        ]
        print(cmd)
        result = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if result.returncode != 0:
            print("ERROR:", result.stderr)
            return False

        return True

    except Exception as e:
        print("EXCEPTION:", e)
        return False

results = [download_video(v) for v in tqdm(video_ids[22:])]
print("Downloaded videos:", sum(results))

  0%|          | 0/22 [00:00<?, ?it/s]

['yt-dlp', 'https://www.youtube.com/watch?v=ChjeZ4a7', '--extract-audio', '--audio-format', 'wav', '--output', 'videos/ChjeZ4a7.wav']


  5%|▍         | 1/22 [00:00<00:09,  2.15it/s]

ERROR: ERROR: [youtube:truncated_id] ChjeZ4a7: Incomplete YouTube ID ChjeZ4a7. URL https://www.youtube.com/watch?v=ChjeZ4a7 looks truncated.

['yt-dlp', 'https://www.youtube.com/watch?v=7jVEKYMcy6o', '--extract-audio', '--audio-format', 'wav', '--output', 'videos/7jVEKYMcy6o.wav']


  9%|▉         | 2/22 [00:03<00:36,  1.84s/it]

...

['yt-dlp', 'https://www.youtube.com/watch?v=VGSeCbxNuoA', '--extract-audio', '--audio-format', 'wav', '--output', 'videos/VGSeCbxNuoA.wav']


 95%|█████████▌| 21/22 [03:15<00:07,  7.75s/it]

['yt-dlp', 'https://www.youtube.com/watch?v=ot1w', '--extract-audio', '--audio-format', 'wav', '--output', 'videos/ot1w.wav']


100%|██████████| 22/22 [03:15<00:00,  8.90s/it]

ERROR: ERROR: [youtube:truncated_id] ot1w: Incomplete YouTube ID ot1w. URL https://www.youtube.com/watch?v=ot1w looks truncated.

Downloaded videos: 20


In [56]:
dataset[0]['utt_id']

'jC2Q_T89YXI-268552-240'

In [66]:
def find_video_file(video_id):
    for f in os.listdir("videos"):
        if f.startswith(video_id):
            return os.path.join("videos", f)
    return None


def cut_audio(item):
    video_id = item["utt_id"][:11]
    offset = float(item["offset"])
    duration = float(item["duration"])
    out_path = item["audio_filepath"]

    video_file = find_video_file(video_id)
    if video_file is None:
        return False

    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    if os.path.exists(out_path):
        return True

    try:
        cmd = [
            "ffmpeg",
            "-y",
            "-loglevel", "info",
            "-i", video_file,
            "-ss", str(offset),
            "-t", str(duration),
            "-ar", "16000",
            "-ac", "1",
            out_path
        ]
        # print(cmd)
        result = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if result.returncode != 0:
            print("ERROR:", result.stderr)
            return False

        return True

    except Exception as e:
        print("EXCEPTION:", e)
        return False

In [67]:
with ThreadPoolExecutor(max_workers=16) as ex:
    results = list(tqdm(ex.map(cut_audio, dataset), total=len(dataset)))

100%|██████████| 2848/2848 [00:00<00:00, 19274.67it/s]


In [68]:
print("Segments created:", sum(results))

Segments created: 71


In [38]:
print("Segments created:", sum(results))

Segments created: 1597
